In [1]:
from neo4j import GraphDatabase
import json 
import uuid

driver=GraphDatabase.driver("neo4j://127.0.0.1:7687",auth=("neo4j","password"))


In [2]:
def load_data(file_path):
    with open(file_path,'r') as f:
        return json.load(f)

def flatten_message(data):
    messages=[]

    def add_messages(msg, source, doc_id=None):
        msg_id = str(uuid.uuid4())
    
        # Ensure msg["data"] is always a list of dicts
        raw_data = msg.get("data", [])
        data_list = raw_data if isinstance(raw_data, list) else [raw_data] if isinstance(raw_data, dict) else []

        messages.append({
            "id": msg_id,
            "content": msg.get("content", "").strip(),
            "role": msg.get("role", "user"),
            "timestamp": msg.get("timestamp", None),
            "topics": [d["topic"] for d in data_list if isinstance(d, dict) and "topic" in d],
            "tables": [d.get("payload") for d in data_list if isinstance(d, dict) and d.get("topic") == "table"],
            "doc_id": doc_id,
            "source": source
        })
        return msg_id
    
    
    #for vcc data
    for doc_id,items in data["old_chat"]["vcc_data"].items():
        last_msg=None
        for entry in items:
            new_msg=add_messages(entry,source="vcc",doc_id=doc_id)
            if last_msg:
                messages.append(("next",last_msg,new_msg))
            last_msg=new_msg

    #for content_data
    for doc_id,items in data["old_chat"]["content_data"].items():
        last_msg=None
        new_msg=add_messages(entry,source="content_data",doc_id=doc_id)
        if last_msg:
            messages.append(("next",last_msg,new_msg))
        last_msg=new_msg

    #for smb_data
    last_msg=None
    for entry in data["old_chat"]["smb_data"]:
        new_msg=add_messages(entry,source="smb")
        if last_msg:
            messages.append(("next",last_msg,new_msg))
        last_msg=new_msg
    return messages
                  


In [4]:
def save_to_neo4j(messages):
    with driver.session() as session:
        for msg in messages:
            if isinstance(msg, tuple) and msg[0] == "next":
                _, from_id, to_id = msg
                session.run("""
                    MATCH (a:Message {id: $from_id}), (b:Message {id: $to_id})
                    MERGE (a)-[:NEXT]->(b)
                """, from_id=from_id, to_id=to_id)
            else:
                session.run("""
                    MERGE (m:Message {id: $id})
                    SET m.content = $content,
                        m.role = $role,
                        m.timestamp = $timestamp,
                        m.source = $source
                """, msg)
                for topic in msg["topics"]:
                    session.run("""
                        MERGE (t:Topic {name: $topic})
                        WITH t
                        MATCH (m:Message {id: $msg_id})
                        MERGE (m)-[:HAS_TOPIC]->(t)
                    """, topic=topic, msg_id=msg["id"])

                for table in msg["tables"]:
                    session.run("""
                        MERGE (tb:Table {title: $title})
                        WITH tb
                        MATCH (m:Message {id: $msg_id})
                        MERGE (m)-[:CONTAINS_TABLE]->(tb)
                    """, title=table["title"], msg_id=msg["id"])

                if msg.get("doc_id"):
                    session.run("""
                        MERGE (d:Document {id: $doc_id})
                        WITH d
                        MATCH (m:Message {id: $msg_id})
                        MERGE (m)-[:REFERS_TO]->(d)
                    """, doc_id=msg["doc_id"], msg_id=msg["id"])

In [5]:
data = load_data("main_data.json")
messages = flatten_message(data)
save_to_neo4j(messages)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
NEO4J_URL="neo4j://127.0.0.1:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="password"


In [3]:
from langchain_community.graphs import Neo4jGraph


In [4]:
graph=Neo4jGraph(url=NEO4J_URL,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)

/tmp/ipykernel_59819/389170914.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph=Neo4jGraph(url=NEO4J_URL,username=NEO4J_USERNAME,password=NEO4J_PASSWORD)


In [5]:
import json

with open("main_data.json","r") as f:
    data=json.load(f)

In [11]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain.chat_models import ChatOpenAI

llm=ChatOpenAI(model="gpt-4o-mini")
transformer=LLMGraphTransformer(llm=llm)



In [10]:
%pip install json-repair

Note: you may need to restart the kernel to use updated packages.


In [12]:
import openai

In [13]:
def extract_triples(text):
    prompt = f"""
    Extract subject-relationship-object triples from the text below.
    Format each triple as (subject, predicate, object).

    TEXT:
    """
    prompt += text + "\n\nTriples:"

    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [14]:
def save_triple(tx, subject, predicate, obj):
    tx.run(
        """
        MERGE (s:Entity {name: $subject})
        MERGE (o:Entity {name: $object})
        MERGE (s)-[r:RELATION {type: $predicate}]->(o)
        """,
        subject=subject,
        predicate=predicate,
        object=obj
    )

In [21]:
def process_text_block(text):
    triples_text = extract_triples(text)
    lines = triples_text.split("\n")
    graph_docs = []

    for line in lines:
        if line.startswith("(") and line.endswith(")"):
            try:
                subject, predicate, obj = line[1:-1].split(",", 2)
                subject = subject.strip()
                predicate = predicate.strip()
                obj = obj.strip()
                graph_docs.append(GraphDocument(
                    nodes=[{"id": subject}, {"id": obj}],
                    relationships=[{
                        "source": subject,
                        "target": obj,
                        "type": predicate
                    }]
                ))
            except Exception as e:
                print(f"Parse failed: {line} — {e}")

    if graph_docs:
        graph.add_graph_documents(graph_docs)

In [22]:
def main():
    with open("main_data.json", "r") as f:
            full_data = json.load(f)

            vcc_data = full_data["old_chat"]["vcc_data"]

            for group_key, content_blocks in vcc_data.items():
                for entry in content_blocks:
                    text = entry.get("content", "")
                    data_field = entry.get("data", {})

                    if isinstance(data_field, dict):
                        process_text_block(text)

                    elif isinstance(data_field, list):
                        for item in data_field:
                            payload = item.get("payload", {})
                            if isinstance(payload, str):
                                process_text_block(payload)
                            elif isinstance(payload, dict):
                                if "title" in payload:
                                    process_text_block(payload["title"])
                                if "rows" in payload:
                                    for row in payload["rows"]:
                                        row_text = " ".join(map(str, row))
                                        process_text_block(row_text)

In [23]:
main()
print("done with the work")

Parse failed: (Shinghals types, have, durability) — name 'GraphDocument' is not defined
Parse failed: (Charcoal, is, a versatile neutral shade) — name 'GraphDocument' is not defined


KeyboardInterrupt: 

In [ ]:
%pip install openai==0.28

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 KB 709.8 kB/s eta 0:00:00 0:00:01
  Attempting uninstall: openai
    Found existing installation: openai 1.77.0
    Uninstalling openai-1.77.0:
      Successfully uninstalled openai-1.77.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.16 requires openai<2.0.0,>=1.68.2, but you have openai 0.28.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain.graphs import Neo4jGraph
from langchain.chat_models import ChatOpenAI
import json


In [4]:
graph1=Neo4jGraph(
    url="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="password"
)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
transformer = LLMGraphTransformer(llm=llm)

/tmp/ipykernel_65093/2928975086.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [8]:
from langchain.schema import Document

In [9]:
with open("main_data.json", "r") as f:
    data = json.load(f)

vcc_data = data["old_chat"]["vcc_data"]

graph_docs = []

for group_id, blocks in vcc_data.items():
    for entry in blocks:
        content = entry.get("content", "")
        data_field = entry.get("data", {})

        if content.strip():
            try:
                docs = transformer.convert_to_graph_documents(
                    [Document(page_content=content)]
                )
                graph_docs.extend(docs)
            except Exception as e:
                print(f"[content] Failed for group {group_id}: {e}")

        # If data is a dict, make it a list for uniformity
        if isinstance(data_field, dict):
            data_field = [data_field]

        for item in data_field:
            payload = item.get("payload", {})
            if isinstance(payload, dict):
                if "title" in payload:
                    try:
                        docs = transformer.convert_to_graph_documents(
                            [Document(page_content=payload["title"])]
                        )
                        graph_docs.extend(docs)
                    except Exception as e:
                        print(f"[title] Failed: {e}")
                if "rows" in payload:
                    for row in payload["rows"]:
                        try:
                            row_text = " ".join(str(cell) for cell in row)
                            docs = transformer.convert_to_graph_documents(
                                [Document(page_content=row_text)]
                            )
                            graph_docs.extend(docs)
                        except Exception as e:
                            print(f"[row] Failed: {e}")


In [11]:
graph1.add_graph_documents(graph_docs)
print(f"✅ Uploaded {len(graph_docs)} knowledge triples to Neo4j.")


✅ Uploaded 50 knowledge triples to Neo4j.
